<a href="https://colab.research.google.com/github/RusakovaAlla/bask_misc/blob/main/sent_emails_from_CXhub.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
"""
Будем парсить сайт https://check-mail.org/domain/ - минимум графики, структурированный ответ по каждому домену
Исходный файл со списком доменов дают, но его нужно будет предварительно почистить
Все домены из файла спарсить не получится: часть запросов заблокирует сам сайт (можно попробовать повторно пройтись), часть почт объединена в строки,
которые мы разделить в рамках исследования не можем, т.к. в файле по ним агрегированные данные.
Поэтому частично придется доделывать позже руками.
Скрипт будет отрабатывать относительно долго - "хакаем" этично, не перегружая запросами сервер, поэтому специально "тормозим" скрипт
"""

In [ ]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
from time import sleep
from random import randint
from datetime import date
from google.colab import files
import re
import json
import logging

Переменные для работы

In [ ]:
#парсим этот, но можно будет взять другой, но состав колонок придется пересмотреть
site = "https://check-mail.org/domain/"
# файлы для чтения, логирования и скачивания результата, название может быть другим
file_to_read = 'extracted_email_data.csv'
file_to_log = 'domain_check.log'
file_to_download = f'Research_email_domains {date.today().strftime("%Y-%m-%d")}.csv'
#колонка в файле с доменами, указывающая на домены 1 и 2 уровней
col = 'Domains'
#заранее посмотрели список параметров на сайте, которые он отдает по запросу, и выбрали нужные
list_params = ['valid', 'block', 'is_disposable', 'is_email_forwarder','risk', 'text', 'mx_host', 'possible_typo']
#данные для логирования
logging.basicConfig(level = logging.INFO, filename = file_to_log, filemode ='w', format= '%(asctime)s %(levelname)s %(message)s', force=True)

In [ ]:
def get_check_mail_domain_data(table, row_index, col_name):
  """
  функция парсинга и обработки данных. Если сайт берем другой, методику обработки нужно будет менять
  """
  domain_data = requests.get(f'{site}{table.loc[row_index,col_name]}')
  site_page = BeautifulSoup(domain_data.content, 'html.parser')
  all_data = site_page.find("div",("class","relative mx-auto")).find("pre").get_text(strip=True) #находим нужный блок с ответом по домену
  #уберем лишние знаки и преобразуем в словарь
  all_data = all_data.replace('\n','')
  all_data = re.sub(r"\s+", " ", all_data)
  all_data = json.loads(all_data)
  result_list = []
  #обработаем ранее сформированный список параметров - аккуратно с 'possible_typo', в нем могут лежать сразу несколько доменов, впишем красиво через резделитель
  for param in list_params:
    if param == 'possible_typo':
      result_list.append(','.join(all_data[param]))
    else:
      result_list.append(all_data[param])

  return result_list


Обрабатываем исходный файл

In [ ]:
start_table = pd.read_csv(file_to_read) #читаем файл
result_table = start_table.loc[:, start_table.columns == col] #берем только нужный столбец
#чистим от лишних знаков - по какой-то причине в ранее передаваемых мне файлах последний знак в строке точка с запятой, это нам не нужно
result_table.loc[:,col] = result_table[col].str[:-1]
#в эту же таблицу будем складывать получаемые данные, добавим нужные столбцы
result_table = result_table.reindex(columns=result_table.columns.tolist() + list_params, fill_value=pd.NA)
#ищем строки, которые не сможем обработать никак - те самые строки, где несколько доменов. Их выделим и позже добавим для разметки вручную
result_table.loc[:,'is_combined'] = result_table.loc[:,col].str.contains(',')# False - оставляем для использования в скрипте

In [ ]:
#True - не обрабатываем, позже сразу добавим в результирующую таблицу
values_to_add_manually = result_table.loc[result_table["is_combined"]==True,result_table.columns != 'is_combined'].reset_index(drop=True)

In [ ]:
#с этой частью таблицы будем дальше работать
result_table = pd.DataFrame(result_table.loc[result_table["is_combined"]==False,result_table.columns != 'is_combined']).reset_index(drop=True)
result_table

,Domains,valid,block,is_disposable,is_email_forwarder,risk,text,mx_host,possible_typo
0,bscse.okcx.edu.rs,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
1,mail.ru,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
2,nondon.store,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
3,pindush.net,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
4,pro.zikzak.site,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
...,...,...,...,...,...,...,...,...,...
1911,xmail.com,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
1912,xn--ndex-43d7i.ru,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
1913,yandeex.ru,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
1914,yandek.ru,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>


In [ ]:
#соберем цикл для обработки данных по доменам
count = 0
for i in range(0, result_table.shape[0]):
  count += 1
  aquired_data = get_check_mail_domain_data(result_table, i, col)
  if not aquired_data:
    logging.info(f'Не удалось получить данные {result_table[i, col]}')
    continue
  else:
    result_table.loc[result_table[col]==result_table.loc[i,col],list_params]=get_check_mail_domain_data(result_table, i, col)
    logging.info(f'ok - {result_table.loc[i, col]}')
  #притормозим скрипт на рандомный период после каждой итерации и через оределенное количество обработанных строк, чтобы сайт не ругался за парсинг
  if count%5 == 0:
    sleep(randint(5,10))
  elif count%111 == 0:
    sleep(randint(10,20))
    #для красоты текста в консоли - чтобы понимали, что что-то происходит
    if count%10 == 1:
      log_pretty_endings = ('а', 'а')
    elif count%10 in (2,3,4):
      log_pretty_endings = ('ы', 'и')
    else:
      log_pretty_endings = ('о', '')
    print(f'Обработан{log_pretty_endings[0]} {count} строк{log_pretty_endings[1]}')
  else:
    sleep(randint(2,5))


Обработана 1 строка
Обработаны 2 строки
Обработаны 3 строки
Обработаны 4 строки


Сборка единого файла для последующей ручной обработки

In [ ]:
#дописываем к результирующему фрейму часть, которую мы не обрабатывали
result_table = pd.concat([result_table, values_to_add_manually], axis=0).reset_index()
result_table

,index,Domains,valid,block,is_disposable,is_email_forwarder,risk,text,mx_host,possible_typo
0,0,bscse.okcx.edu.rs,True,True,True,False,91,Disposable / temporary domain,mail.kuromee.com,
1,1,mail.ru,True,False,False,False,5,Looks okay,mxs.mail.ru,
2,2,nondon.store,True,True,True,False,100,Disposable / temporary domain,mail.kuromee.com,
3,3,pindush.net,True,True,True,False,91,Disposable / temporary domain,mail.kuromee.com,
4,4,pro.zikzak.site,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
...,...,...,...,...,...,...,...,...,...,...
2030,114,"di.binitrax.com;, di.viremond.com",<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
2031,115,"az.aurenics.com;, az.topkcorp.com",<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
2032,116,"Hotmail.com;, hotmail.ru",<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
2033,117,"dk.duskilon.com;, dk.ru",<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>


In [ ]:
#записываем и скачиваем файл, для последующей дообработки, но уже вручную
result_table.to_csv(file_to_download, index=False)
files.download(file_to_download)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>